In [5]:
import pandas as pd
df = pd.DataFrame()

#new code

In [10]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import random

# Setup undetected Chrome
options = uc.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = uc.Chrome(version_main=140, options=options)

records = []

pages = list(range(999, 1002))  
random.shuffle(pages)   # shuffle to avoid pattern detection

for page_id in pages:
    url = f"https://www.yellowpages.my/services/l?page={page_id}"

    # random sleep sebelum load page
    time.sleep(random.randint(5, 12))

    print(f"Scraping page {page_id} ...")
    driver.get(url)

    # function to try extract company info
    def extract_cards():
        try:
            cards = WebDriverWait(driver, 20).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a.title"))
            )
        except:
            cards = []

        page_records = []
        for card in cards:
            try:
                name = card.text.strip()
                parent = card.find_element(By.XPATH, "..")
                try:
                    address = parent.find_element(By.CSS_SELECTOR, ".location-group").text.strip()
                except:
                    address = None


                page_records.append({
                    "page": page_id,
                    "Company Name": name,
                    "Address": address

                    
                })
            except:
                continue
        return page_records

    # First try
    page_data = extract_cards()

    # If less than 12 results, retry once
    if len(page_data) < 12:
        print(f"⚠️ Page {page_id} only got {len(page_data)} records, retrying...")
        driver.get(url)
        time.sleep(random.randint(5, 12))
        page_data = extract_cards()

    # Save the data
    records.extend(page_data)

    # Save progressively 
    df = pd.DataFrame(records)
    df.to_csv("yellowpages_companies_try1.csv", index=False, encoding="utf-8-sig")

driver.quit()

print(f"✅ Scraped {len(records)} records total")


Scraping page 1001 ...
Scraping page 1000 ...
Scraping page 999 ...
✅ Scraped 36 records total


In [11]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import random

# Setup undetected Chrome
options = uc.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = uc.Chrome(version_main=140, options=options)

records = []

pages = list(range(999, 1002))  
random.shuffle(pages)

for page_id in pages:
    url = f"https://www.yellowpages.my/services/l?page={page_id}"

    time.sleep(random.randint(5, 12))  # random sleep
    print(f"Scraping page {page_id} ...")
    driver.get(url)

    def extract_cards():
        try:
            cards = WebDriverWait(driver, 20).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a.title"))
            )
        except:
            cards = []

        page_records = []
        for card in cards:
            try:
                name = card.text.strip()

                # cari container untuk satu company
                container = card.find_element(By.XPATH, "./ancestor::div[contains(@class,'company-card')]")

                # extract semua div & span address dalam container
                try:
                    addr_divs = container.find_elements(By.CSS_SELECTOR, "div.ng-star-inserted")
                    addr_spans = container.find_elements(By.CSS_SELECTOR, "span.ng-star-inserted")

                    # gabungkan semua text
                    parts = [el.text.strip() for el in addr_divs + addr_spans if el.text.strip()]
                    address = ", ".join(parts) if parts else None
                except:
                    address = None

                page_records.append({
                    "page": page_id,
                    "Company Name": name,
                    "Address": address
                })
            except:
                continue
        return page_records

    # First try
    page_data = extract_cards()

    # Retry sekali kalau result sikit sangat
    if len(page_data) < 12:
        print(f"⚠️ Page {page_id} only got {len(page_data)} records, retrying...")
        driver.get(url)
        time.sleep(random.randint(5, 12))
        page_data = extract_cards()

    # Simpan data
    records.extend(page_data)

    # Save progressively
    df = pd.DataFrame(records)
    df.to_csv("yellowpages_companies_try1.csv", index=False, encoding="utf-8-sig")

driver.quit()

print(f"✅ Scraped {len(records)} records total")


Scraping page 1000 ...
⚠️ Page 1000 only got 0 records, retrying...
Scraping page 1001 ...
⚠️ Page 1001 only got 0 records, retrying...
Scraping page 999 ...
⚠️ Page 999 only got 0 records, retrying...
✅ Scraped 0 records total


In [ ]:
df = pd.DataFrame(records)
df.to_csv("yellowpages_companies.csv", index=False, encoding="utf-8-sig")

#combine csv

In [98]:
import pandas as pd

# read CSV
df1 = pd.read_csv("yellowpages_companies0.csv")
df2 = pd.read_csv("yellowpages_companies1.csv")
df3 = pd.read_csv("yellowpages_companies2.csv")
df4 = pd.read_csv("yellowpages_companies3.csv")
df5 = pd.read_csv("yellowpages_companies4.csv")
df6 = pd.read_csv("yellowpages_companies5.csv")
df7 = pd.read_csv("yellowpages_companies6.csv")
df8 = pd.read_csv("yellowpages_companies7.csv")
df9 = pd.read_csv("yellowpages_companies8.csv")
df10 = pd.read_csv("yellowpages_companies9.csv")
df11 = pd.read_csv("yellowpages_companies10.csv")


# combine all dataframes
combined = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11], ignore_index=True)

# save combined dataframe to CSV
combined.to_csv("yellowpages_combined.csv", index=False, encoding="utf-8-sig")

print("Done! Total rows:", len(combined))


Done! Total rows: 239988


#check missing pages

In [99]:
import pandas as pd

# read CSV
df1 = pd.read_csv("yellowpages_mashita1.csv")
df2 = pd.read_csv("yellowpages_combined9.csv")
df3 = pd.read_csv("yellowpages_combined10.csv")




# combine all dataframes
combined = pd.concat([df1, df2, df3], ignore_index=True)

# save combined dataframe to CSV
combined.to_csv("yellowpages_mashita.csv", index=False, encoding="utf-8-sig")

print("Done! Total rows:", len(combined))

Done! Total rows: 120012


In [ ]:
import pandas as pd

# read CSV
df = pd.read_csv("yellowpages_combined2.csv")

# Group by page
counts = df.groupby("page").size()

# Detect pages tak cukup 12
missing_pages = counts[counts < 12]

print(f"Total pages scraped: {len(counts)}")
print(f"Pages with missing records: {len(missing_pages)}")

# List out details
print(missing_pages)

# Check kalau ada page langsung tak keluar
all_pages = set(range(40001, 40732))
scraped_pages = set(df["page"].unique())
not_scraped = all_pages - scraped_pages

print(f"Pages not scraped at all: {len(not_scraped)}")
print(not_scraped)


Total pages scraped: 731
Pages with missing records: 1
page
40731    4
dtype: int64
Pages not scraped at all: 0
set()


In [105]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import time, random

# Load CSV yang dah ada
df_existing = pd.read_csv("yellowpages_combined1.csv")
scraped_pages = set(df_existing["page"].unique())

# Range asal
all_pages = set(range(40001, 40732))

# Cari missing pages
not_scraped = sorted(all_pages - scraped_pages)
print(f"Missing pages: {len(not_scraped)}")

# Setup Selenium
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

records = []

for page_id in not_scraped:
    url = f"https://www.yellowpages.my/services/l?page={page_id}"
    driver.get(url)
    time.sleep(random.randint(5, 15))  # bagi masa load

    cards = driver.find_elements(By.CSS_SELECTOR, "a.title")

    for card in cards:
        try:
            name = card.text.strip()
            parent = card.find_element(By.XPATH, "..")
            try:
                address = parent.find_element(By.CSS_SELECTOR, ".location-group").text.strip()
            except:
                address = None

            records.append({
                "page": page_id,
                "Company Name": name,
                "Address": address
            })
        except Exception as e:
            print("Error:", e)

# Save missing pages result
df_missing = pd.DataFrame(records)
df_missing.to_csv("yellowpages_missing.csv", index=False, encoding="utf-8-sig")

driver.quit()
print(f"Scraped {len(records)} records from missing pages ✅")


Missing pages: 100
Scraped 1192 records from missing pages ✅


In [110]:
df1 = pd.read_csv("yellowpages_mashita.csv")
df2 = pd.read_csv("yellowpages_combined2.csv")

df_all = pd.concat([df1, df2], ignore_index=True)
df_all.to_csv("yellowpages_combined3.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd
import glob

# Baca semua CSV (3 file)
files = ["yellowpages.csv", "yellowpages_ameer.csv", "yellowpages_combined3.csv"]
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

num_parts = 4
chunk_size = len(df) // num_parts + 1

# Pecahkan jadi 4 file
for i in range(num_parts):
    start = i * chunk_size
    end = start + chunk_size
    chunk = df.iloc[start:end]
    chunk.to_csv(f"data_part_{i+1}.csv", index=False)

print("Done.")


Siap! Dah jadi 4 file seimbang.
